# Data Preparation

This notebook builds the complete data pipeline for a low-resource Arabic to English machine translation project.

We download the IWSLT 2017 dataset, clean and normalize the text, filter out problematic sentence pairs, and save the final dataset in multiple formats.

Pipeline Overview
- Download: Fetch IWSLT 2017 from Hugging Face
- Normalize: Light Arabic and English normalization to reduce vocabulary noise
- Filter: Remove empty, too long, and bad ratio pairs to keep only trainable pairs
- Deduplicate: Remove repeated pairs in train to prevent memorization
- Sample: Take 50,000 random training pairs to simulate a low-resource setting
- Save: Plain text, CSV, and Parquet for different needs


## 1. Setup

Import libraries and define all constants upfront so nothing is hidden later.


In [ ]:
from pathlib import Path
import random, re
from datasets import load_dataset
import pandas as pd
import numpy as np

print('Imports loaded successfully')


Imports loaded successfully


### Constants

These values control the entire cleaning pipeline. Changing them here changes everything downstream.

- 50,000 pairs: intentionally small to simulate a low resource scenario
- 80 tokens max: longer sentences are harder for small models and add noise
- 3.0 ratio: if one side is 3x longer than the other, the alignment is probably bad


In [ ]:
DATASET_ID        = 'IWSLT/iwslt2017'
DATASET_CONFIG    = 'iwslt2017-ar-en'
SOURCE_LANG       = 'ar'
TARGET_LANG       = 'en'
TRAIN_SUBSET_SIZE = 50_000
RANDOM_SEED       = 42
MAX_SOURCE_TOKENS = 80
MAX_TARGET_TOKENS = 80
MAX_LENGTH_RATIO  = 3.0

print(f'Dataset       : {DATASET_ID} ({DATASET_CONFIG})')
print(f'Direction     : {SOURCE_LANG} -> {TARGET_LANG}')
print(f'Train subset  : {TRAIN_SUBSET_SIZE:,}')
print(f'Max tokens    : {MAX_SOURCE_TOKENS}')
print(f'Max ratio     : {MAX_LENGTH_RATIO}')


Dataset       : IWSLT/iwslt2017 (iwslt2017-ar-en)
Direction     : ar -> en
Train subset  : 50,000
Max tokens    : 80
Max ratio     : 3.0


## 2. Download Dataset

IWSLT 2017 contains TED talk transcripts aligned across multiple languages. We use the Arabic to English configuration.

The dataset has three pre-defined splits:
- train: ~231k pairs (we will sample 50k from these)
- validation: 888 pairs (for monitoring training loss)
- test: ~8.5k pairs (for final evaluation)

Note: The dataset requires trust_remote_code=True because it uses a custom loading script.


In [ ]:
try:
    dataset = load_dataset(DATASET_ID, DATASET_CONFIG)
except Exception:
    print('Retrying with trust_remote_code=True')
    dataset = load_dataset(DATASET_ID, DATASET_CONFIG, trust_remote_code=True)

for split in dataset:
    print(f'{split:12s} : {len(dataset[split]):>8,} pairs')


Using the latest cached version of the dataset since IWSLT/iwslt2017 couldn't be found on the Hugging Face Hub


Found the latest cached dataset configuration 'iwslt2017-ar-en' at C:\Users\Ahmed\.cache\huggingface\datasets\IWSLT___iwslt2017\iwslt2017-ar-en\1.0.0\03ce9110373117c6f6687719f49f269486a8cd49dcad2527993a316cd4b6ad49 (last modified on Wed May 13 10:37:11 2026).


train        :  231,713 pairs
test         :    8,583 pairs
validation   :      888 pairs


### Preview

Let's look at one example to understand the data format.
Each item has a translation dict with ar and en keys.


In [ ]:
example = dataset['train'][0]

print('Arabic:')
print(example['translation']['ar'])
print()
print('English:')
print(example['translation']['en'])


Arabic:
شكرا جزيلا كريس. انه فعلا شرف عظيم لي

English:
Thank you so much, Chris.


## 3. Save Raw Files

Before any cleaning, we save the original text as-is. This gives us a reference point if we ever need to check what the data looked like before normalization.

Format: one sentence per line, UTF-8 encoded.
Any newlines embedded inside a sentence are replaced with spaces.


In [ ]:
for split in dataset:
    ar_path = f'../data/raw/{split}.ar.raw'
    en_path = f'../data/raw/{split}.en.raw'

    with open(ar_path, 'w', encoding='utf-8') as f_ar, \
         open(en_path, 'w', encoding='utf-8') as f_en:
        for item in dataset[split]:
            ar = item['translation']['ar'].replace('\n', ' ').strip()
            en = item['translation']['en'].replace('\n', ' ').strip()
            f_ar.write(ar + '\n')
            f_en.write(en + '\n')

    print(f'Saved: {ar_path}  |  {en_path}')


Saved: ../data/raw/train.ar.raw  |  ../data/raw/train.en.raw
Saved: ../data/raw/test.ar.raw  |  ../data/raw/test.en.raw
Saved: ../data/raw/validation.ar.raw  |  ../data/raw/validation.en.raw


In [ ]:
for split in dataset:
    with open(f'../data/raw/{split}.ar.raw', 'r', encoding='utf-8') as f:
        n_ar = sum(1 for _ in f)
    with open(f'../data/raw/{split}.en.raw', 'r', encoding='utf-8') as f:
        n_en = sum(1 for _ in f)

    status = 'OK' if n_ar == n_en else 'MISMATCH!'
    print(f'{split:12s} : AR={n_ar:,}  EN={n_en:,}  {status}')


train        : AR=231,713  EN=231,713  OK
test         : AR=8,583  EN=8,583  OK
validation   : AR=888  EN=888  OK


## 4. Build DataFrames

Load the raw files into pandas DataFrames for easier manipulation.
From this point on, we work with DataFrames instead of the Hugging Face dataset object.


In [ ]:
# Load raw text into DataFrames 
dfs = {}

for split in ['train', 'validation', 'test']:
    with open(f'../data/raw/{split}.ar.raw', 'r', encoding='utf-8') as f:
        ar = [line.rstrip('\n') for line in f]
    with open(f'../data/raw/{split}.en.raw', 'r', encoding='utf-8') as f:
        en = [line.rstrip('\n') for line in f]

    dfs[split] = pd.DataFrame({'split': split, 'source_ar': ar, 'target_en': en})
    print(f'{split:12s} : {dfs[split].shape}')


train        : (231713, 3)
validation   : (888, 3)
test         : (8583, 3)


## 5. Text Normalization

We apply light normalization, just enough to reduce noise without losing meaning.

Arabic Normalization
Arabic has multiple forms of the same letter that mean the same thing:
- alef variants to plain alef
- alef maqsura to ya

This prevents the model from treating different spellings of the same word as different words.

English Normalization
- Lowercase everything
- Collapse multiple spaces into one


In [ ]:
# Define normalization functions 
def normalize_arabic_light(text):
    text = re.sub(r'[\u0623\u0625\u0622]', '\u0627', text)
    text = re.sub(r'\u0649', '\u064a', text)
    return re.sub(r'\s+', ' ', text).strip()

def normalize_english_light(text):
    return re.sub(r'\s+', ' ', text.lower()).strip()

print('Normalization functions defined')


Normalization functions defined


In [ ]:
# Apply normalization 
for split in dfs:
    dfs[split]['source_ar_clean'] = dfs[split]['source_ar'].apply(normalize_arabic_light)
    dfs[split]['target_en_clean'] = dfs[split]['target_en'].apply(normalize_english_light)

# Show before/after
ex = dfs['train'].iloc[0]
print('Arabic before:', ex['source_ar'])
print('Arabic after :', ex['source_ar_clean'])
print()
print('English before:', ex['target_en'])
print('English after :', ex['target_en_clean'])

Arabic before: شكرا جزيلا كريس. انه فعلا شرف عظيم لي
Arabic after : شكرا جزيلا كريس. انه فعلا شرف عظيم لي

English before: Thank you so much, Chris.
English after : thank you so much, chris.


## 6. Compute Token Lengths

We count tokens using simple whitespace splitting.
This is approximate, especially for Arabic, but good enough for filtering.

The length ratio catches misaligned pairs. If one side is much longer than the other, the translation is probably broken.


In [ ]:
# Add token counts and length ratio 
for split in dfs:
    df = dfs[split]
    df['source_len'] = df['source_ar_clean'].apply(lambda t: len(t.split()) if t.strip() else 0)
    df['target_len'] = df['target_en_clean'].apply(lambda t: len(t.split()) if t.strip() else 0)
    df['length_ratio'] = df.apply(
        lambda r: max(r['source_len']/r['target_len'], r['target_len']/r['source_len'])
        if min(r['source_len'], r['target_len']) > 0 else float('inf'), axis=1
    )

print('Train length stats:')
print(dfs['train'][['source_len', 'target_len', 'length_ratio']].describe().round(2))


Train length stats:
       source_len  target_len  length_ratio
count   231713.00   231713.00     231713.00
mean        14.33       17.37          1.31
std         10.15       12.24          0.29
min          1.00        1.00          1.00
25%          7.00        9.00          1.12
50%         12.00       14.00          1.25
75%         19.00       23.00          1.42
max         82.00       89.00         11.00


## 7. Filter Bad Pairs

Three types of sentence pairs are removed from all splits:

- Empty: source or target has 0 tokens, because there is nothing to learn from
- Too long: either side is over 80 tokens, because small models can't handle them
- Bad ratio: the ratio is over 3.0, because they are likely misaligned


In [ ]:
# Filter empty pairs 
removed_list = []

for split in dfs:
    mask = (dfs[split]['source_len'] == 0) | (dfs[split]['target_len'] == 0)
    removed = dfs[split][mask].copy()
    if len(removed) > 0:
        removed['reason'] = 'empty_pair'
        removed_list.append(removed)
    dfs[split] = dfs[split][~mask].reset_index(drop=True)
    print(f'{split}: removed {mask.sum()} empty -> {len(dfs[split]):,} remaining')


train: removed 0 empty -> 231,713 remaining
validation: removed 0 empty -> 888 remaining
test: removed 0 empty -> 8,583 remaining


In [ ]:
# Filter too-long pairs (> 80 tokens) 
for split in dfs:
    mask = (dfs[split]['source_len'] > MAX_SOURCE_TOKENS) | (dfs[split]['target_len'] > MAX_TARGET_TOKENS)
    removed = dfs[split][mask].copy()
    if len(removed) > 0:
        removed['reason'] = 'too_long'
        removed_list.append(removed)
    dfs[split] = dfs[split][~mask].reset_index(drop=True)
    print(f'{split}: removed {mask.sum()} too-long -> {len(dfs[split]):,} remaining')


train: removed 54 too-long -> 231,659 remaining
validation: removed 0 too-long -> 888 remaining
test: removed 1 too-long -> 8,582 remaining


In [ ]:
# Filter bad length ratios (> 3.0) 
for split in dfs:
    mask = dfs[split]['length_ratio'] > MAX_LENGTH_RATIO
    removed = dfs[split][mask].copy()
    if len(removed) > 0:
        removed['reason'] = 'bad_length_ratio'
        removed_list.append(removed)
    dfs[split] = dfs[split][~mask].reset_index(drop=True)
    print(f'{split}: removed {mask.sum()} bad-ratio -> {len(dfs[split]):,} remaining')


train: removed 309 bad-ratio -> 231,350 remaining
validation: removed 0 bad-ratio -> 888 remaining
test: removed 15 bad-ratio -> 8,567 remaining


## 8. Deduplicate and Sample

Deduplication is applied to training data only. Validation and test are kept as-is because they come from the official dataset splits.

Sampling: We randomly pick exactly 50,000 pairs from the remaining training data.
This simulates a low-resource setting. Having limited data forces the model to generalize better, which is the focus of this project.

We use a fixed random seed for reproducibility, so anyone running this notebook gets the same 50k pairs.


In [ ]:
# Remove duplicate training pairs 
before = len(dfs['train'])
dup_mask = dfs['train'].duplicated(subset=['source_ar_clean', 'target_en_clean'], keep='first')

removed = dfs['train'][dup_mask].copy()
if len(removed) > 0:
    removed['reason'] = 'duplicate_train_pair'
    removed_list.append(removed)

dfs['train'] = dfs['train'][~dup_mask].reset_index(drop=True)
print(f'Removed {dup_mask.sum():,} duplicates: {before:,} -> {len(dfs["train"]):,}')


Removed 1,663 duplicates: 231,350 -> 229,687


In [ ]:
# Randomly sample 50,000 training pairs
available = len(dfs['train'])

if available >= TRAIN_SUBSET_SIZE:
    dfs['train'] = dfs['train'].sample(n=TRAIN_SUBSET_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f'Sampled {TRAIN_SUBSET_SIZE:,} from {available:,} (seed={RANDOM_SEED})')
else:
    print(f'WARNING: only {available:,} pairs available')

print(f'Final train size: {len(dfs["train"]):,}')


Sampled 50,000 from 229,687 (seed=42)
Final train size: 50,000


## 9. Finalize Schema

Replace the original text columns with the cleaned versions and keep only the final columns:
split, source_ar, target_en, source_len, target_len, length_ratio


In [ ]:
# Finalize column schema 
FINAL_COLS = ['split', 'source_ar', 'target_en', 'source_len', 'target_len', 'length_ratio']

for split in dfs:
    df = dfs[split].copy()
    df['source_ar'] = df['source_ar_clean']
    df['target_en'] = df['target_en_clean']
    dfs[split] = df[FINAL_COLS].reset_index(drop=True)
    print(f'{split:12s} : {dfs[split].shape}')


train        : (50000, 6)
validation   : (888, 6)
test         : (8567, 6)


## 10. Save Output Files

We save in three formats because each serves a different purpose:

- Plain text in data/clean for model training (one sentence per line)
- CSV in data/processed for human inspection (open in Excel)
- Parquet in data/processed for fast pandas loading


In [ ]:
# Save clean plain text files 
for split, df in dfs.items():
    for lang, col in [('ar', 'source_ar'), ('en', 'target_en')]:
        path = f'../data/clean/{split}.{lang}'
        with open(path, 'w', encoding='utf-8') as f:
            for text in df[col]:
                f.write(text + '\n')
        print(f'Saved: {path}')


Saved: ../data/clean/train.ar
Saved: ../data/clean/train.en
Saved: ../data/clean/validation.ar
Saved: ../data/clean/validation.en
Saved: ../data/clean/test.ar


Saved: ../data/clean/test.en


In [ ]:
# Save CSV files 
for split, df in dfs.items():
    path = f'../data/processed/{split}_clean.csv'
    df.to_csv(path, index=False, encoding='utf-8')
    print(f'Saved: {path}')

all_df = pd.concat(dfs.values(), ignore_index=True)
all_df.to_csv('../data/processed/all_clean.csv', index=False, encoding='utf-8')
print(f'Saved: ../data/processed/all_clean.csv ({len(all_df):,} rows)')


Saved: ../data/processed/train_clean.csv
Saved: ../data/processed/validation_clean.csv
Saved: ../data/processed/test_clean.csv


Saved: ../data/processed/all_clean.csv (59,455 rows)


In [ ]:
# Save Parquet files 
for split, df in dfs.items():
    path = f'../data/processed/{split}_clean.parquet'
    df.to_parquet(path, index=False)
    print(f'Saved: {path}')

all_df.to_parquet('../data/processed/all_clean.parquet', index=False)
print('Saved: ../data/processed/all_clean.parquet')


Saved: ../data/processed/train_clean.parquet
Saved: ../data/processed/validation_clean.parquet
Saved: ../data/processed/test_clean.parquet
Saved: ../data/processed/all_clean.parquet


In [ ]:
# Save removed pairs for inspection 
if removed_list:
    removed_df = pd.concat(removed_list, ignore_index=True)
    if 'source_ar_clean' in removed_df.columns:
        removed_df['source_ar'] = removed_df['source_ar_clean'].fillna(removed_df['source_ar'])
    if 'target_en_clean' in removed_df.columns:
        removed_df['target_en'] = removed_df['target_en_clean'].fillna(removed_df['target_en'])
    cols = ['split','source_ar','target_en','reason','source_len','target_len','length_ratio']
    for c in cols:
        if c not in removed_df.columns: removed_df[c] = None
    removed_df[cols].to_csv('../data/processed/removed_pairs.csv', index=False, encoding='utf-8')
    print(f'Removed pairs: {len(removed_df):,}')
    print(removed_df['reason'].value_counts().to_string())
    print('Saved: ../data/processed/removed_pairs.csv')


Removed pairs: 2,042
reason
duplicate_train_pair    1663
bad_length_ratio         324
too_long                  55
Saved: ../data/processed/removed_pairs.csv


## 11. Verification

Final sanity checks before moving to the next notebook:
- Arabic and English files have the same number of lines
- No empty lines in any file
- Maximum token length does not exceed 80
- No duplicate pairs in train


In [ ]:
# Final verification 
all_ok = True
for split in ['train', 'validation', 'test']:
    with open(f'../data/clean/{split}.ar', 'r', encoding='utf-8') as f: ar = f.readlines()
    with open(f'../data/clean/{split}.en', 'r', encoding='utf-8') as f: en = f.readlines()

    match = len(ar) == len(en)
    no_empty = all(l.strip() for l in ar) and all(l.strip() for l in en)
    max_ar = max(len(l.split()) for l in ar)
    max_en = max(len(l.split()) for l in en)

    print(f'{split}: AR={len(ar):,} EN={len(en):,} match={match} max_ar={max_ar} max_en={max_en}')
    if not match or not no_empty: all_ok = False
    if max_ar > 80 or max_en > 80:
        print(f'  WARNING: max length > 80!')
        all_ok = False

# Check duplicates in train
with open('../data/clean/train.ar', 'r', encoding='utf-8') as f: t_ar = f.readlines()
with open('../data/clean/train.en', 'r', encoding='utf-8') as f: t_en = f.readlines()
dupes = len(t_ar) - len(set(zip(t_ar, t_en)))
print(f'\nTrain duplicates: {dupes}')
if dupes > 0: all_ok = False

print()
print('All checks passed.' if all_ok else 'WARNING: some checks failed!')


train: AR=50,000 EN=50,000 match=True max_ar=80 max_en=80
validation: AR=888 EN=888 match=True max_ar=66 max_en=77
test: AR=8,567 EN=8,567 match=True max_ar=77 max_en=78



Train duplicates: 0

All checks passed.
